In [56]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from sklearn.preprocessing import StandardScaler

In [28]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yasserh/imdb-movie-ratings-sentiment-analysis")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-movie-ratings-sentiment-analysis' dataset.
Path to dataset files: /kaggle/input/imdb-movie-ratings-sentiment-analysis


In [29]:
import os

df = pd.read_csv(os.path.join(path, "movie.csv"))
df.head()

,text,label
0,I grew up (b. 1965) watching and loving the Th...,0
1,"When I put this movie in my DVD player, and sa...",0
2,Why do people who do not know what a particula...,0
3,Even though I have great interest in Biblical ...,0
4,Im a die hard Dads Army fan and nothing will e...,1


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    40000 non-null  object
 1   label   40000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 625.1+ KB


In [31]:

import string
from sklearn.preprocessing import LabelEncoder
import warnings
import nltk
nltk.download('stopwords') # Download NLTK stopwords
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.metrics.pairwise import cosine_similarity
stop=set(stopwords.words('english'))
warnings.filterwarnings("ignore")
from sklearn.feature_extraction.text import CountVectorizer
import string

stop = set(stopwords.words('english'))

def remove_num(txt):
    new = ''
    for i in str(txt): # Added str() to handle potential non-string data
        if not i.isdigit():
            new = new + i
    return new

def remove_pun(txt):
    # Fixed: The first two arguments should be empty strings, not ' '
    return str(txt).translate(str.maketrans('', '', string.punctuation))

def remove_stopwords(txt):
    word = str(txt).split()
    text = []
    for i in word:
        if i not in stop:
            text.append(i)
    # FIX: You were returning 'txt' (original). You must return " ".join(text)
    return " ".join(text)

def remove_emoji(txt):
    e = ""
    for i in str(txt):
        if i.isascii():
            e = e + i
    return e

def to_lower(txt):
    return str(txt).lower()

import pandas as pd
!pip install langdetect
!pip install deep_translator
from langdetect import detect
from deep_translator import GoogleTranslator

def translate_if_hindi(text):
    """
    Detects if the text is Hindi. If yes, translates to English.
    Otherwise, returns the original text.
    """
    # 1. Handle empty cells or missing data (NaN)
    if pd.isna(text) or not str(text).strip():
        return text

    # Convert to string just in case
    text = str(text)

    try:
        # 2. Detect the language
        # 'hi' is the standard language code for Hindi
        lang = detect(text)

        # 3. Translate if it is Hindi
        if lang == 'hi':
            # Translate from Hindi to English
            translated = GoogleTranslator(source='hi', target='en').translate(text)
            return translated
        else:
            # If it's English, Spanish, etc., leave it alone
            return text

    except Exception as e:
        # This catches errors (e.g., if the text is just numbers or emojis)
        # It will just return the original text without crashing
        return text

def master_clean(txt):
    # If the input is Null/NaN, return an empty string to avoid crashes
    if not isinstance(txt, str) and not isinstance(txt, (int, float)):
        return ""

    txt = to_lower(txt)
    txt = remove_emoji(txt)
    txt = remove_pun(txt)
    txt = remove_num(txt)
    txt = remove_stopwords(txt)
    txt = translate_if_hindi(txt)
    return txt

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [32]:
df['cheaned_text']=df['text'].apply(master_clean)

In [33]:
df.head()

,text,label,cheaned_text
0,I grew up (b. 1965) watching and loving the Th...,0,grew b watching loving thunderbirds mates scho...
1,"When I put this movie in my DVD player, and sa...",0,put movie dvd player sat coke chips expectatio...
2,Why do people who do not know what a particula...,0,people know particular time past like feel nee...
3,Even though I have great interest in Biblical ...,0,even though great interest biblical movies bor...
4,Im a die hard Dads Army fan and nothing will e...,1,im die hard dads army fan nothing ever change ...


In [40]:
df['NER_Cleaned'] = df['NER'].apply(remove_pun)
df['NER_Cleaned'] = df['NER_Cleaned'].apply(to_lower)

In [46]:
df.head()

,text,label,cheaned_text,NER,NER_Cleaned
0,I grew up (b. 1965) watching and loving the Th...,0,grew b watching loving thunderbirds mates scho...,"[(1965, DATE), (Thunderbirds, PRODUCT), (Thund...",1965 date thunderbirds product thunderbirds pr...
1,"When I put this movie in my DVD player, and sa...",0,put movie dvd player sat coke chips expectatio...,"[(first, ORDINAL), (Awsome, PERSON), (Atlantis...",first ordinal awsome person atlantis product f...
2,Why do people who do not know what a particula...,0,people know particular time past like feel nee...,"[(Woodstock, GPE), (the Civil War, EVENT), (Ap...",woodstock gpe the civil war event apollo org t...
3,Even though I have great interest in Biblical ...,0,even though great interest biblical movies bor...,"[(every minute, TIME), (Abraham, PERSON), (Noa...",every minute time abraham person noah person
4,Im a die hard Dads Army fan and nothing will e...,1,im die hard dads army fan nothing ever change ...,"[(Dads Army, ORG), (Man and the hour, WORK_OF_...",dads army org man and the hour workofart battl...


In [47]:
df['all_text']=df['cheaned_text']+df['NER_Cleaned']

In [48]:
df.head()

,text,label,cheaned_text,NER,NER_Cleaned,all_text
0,I grew up (b. 1965) watching and loving the Th...,0,grew b watching loving thunderbirds mates scho...,"[(1965, DATE), (Thunderbirds, PRODUCT), (Thund...",1965 date thunderbirds product thunderbirds pr...,grew b watching loving thunderbirds mates scho...
1,"When I put this movie in my DVD player, and sa...",0,put movie dvd player sat coke chips expectatio...,"[(first, ORDINAL), (Awsome, PERSON), (Atlantis...",first ordinal awsome person atlantis product f...,put movie dvd player sat coke chips expectatio...
2,Why do people who do not know what a particula...,0,people know particular time past like feel nee...,"[(Woodstock, GPE), (the Civil War, EVENT), (Ap...",woodstock gpe the civil war event apollo org t...,people know particular time past like feel nee...
3,Even though I have great interest in Biblical ...,0,even though great interest biblical movies bor...,"[(every minute, TIME), (Abraham, PERSON), (Noa...",every minute time abraham person noah person,even though great interest biblical movies bor...
4,Im a die hard Dads Army fan and nothing will e...,1,im die hard dads army fan nothing ever change ...,"[(Dads Army, ORG), (Man and the hour, WORK_OF_...",dads army org man and the hour workofart battl...,im die hard dads army fan nothing ever change ...


In [49]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
tokenizer = Tokenizer(num_words=10000, oov_token='<unk>')
tokenizer.fit_on_texts(df['all_text'])

In [50]:
text_sequences = tokenizer.texts_to_sequences(df['all_text'])

In [51]:
max_sequence_length = max([len(x) for x in text_sequences])
print(f"Maximum sequence length: {max_sequence_length}")
padded_sequences = pad_sequences(text_sequences, maxlen=max_sequence_length, padding='post')
print(f"Shape of padded sequences: {padded_sequences.shape}")

Maximum sequence length: 2145
Shape of padded sequences: (40000, 2145)


In [52]:
from sklearn.model_selection import train_test_split
X = padded_sequences
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [57]:
vocabulary_size = len(tokenizer.word_index) + 1
embedding_dim = 100

model = Sequential([
    tf.keras.layers.Embedding(input_dim=vocabulary_size, output_dim=embedding_dim, input_length=max_sequence_length),
    Bidirectional(LSTM(units=128, return_sequences=True)), # Using Bidirectional LSTM
    tf.keras.layers.Dropout(0.3),
    Bidirectional(LSTM(units=64)), # Using Bidirectional LSTM
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(units=32, activation='relu'),
    tf.keras.layers.Dense(units=1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [59]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.1
)
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Epoch 1/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 244s 271ms/step - accuracy: 0.8238 - loss: 0.4071 - val_accuracy: 0.8356 - val_loss: 0.3803
Epoch 2/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 246s 273ms/step - accuracy: 0.8920 - loss: 0.2735 - val_accuracy: 0.8647 - val_loss: 0.3221
Epoch 3/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 246s 273ms/step - accuracy: 0.9276 - loss: 0.1944 - val_accuracy: 0.8584 - val_loss: 0.3198
Epoch 4/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 247s 274ms/step - accuracy: 0.9531 - loss: 0.1338 - val_accuracy: 0.8691 - val_loss: 0.3676
Epoch 5/5
900/900 ━━━━━━━━━━━━━━━━━━━━ 247s 274ms/step - accuracy: 0.9656 - loss: 0.0976 - val_accuracy: 0.8634 - val_loss: 0.4352
Test Loss: 0.4304
Test Accuracy: 0.8656


In [60]:
model.save("LSTM.h5")